# 🌿 Cassava Leaf Disease Classification
## Notebook 05: Hyperparameter Tuning

**ML Coursework Project**

---

### Notebook Objectives
1. Use GridSearchCV for Logistic Regression
2. Use GridSearchCV for SVM
3. Use RandomizedSearchCV for Random Forest
4. Document best parameters found
5. Retrain models with optimal parameters

## 1. Import Libraries

In [16]:
# Core libraries
import os
import numpy as np
import pandas as pd
from pathlib import Path
import json
import pickle
import time

# Machine Learning
from sklearn.metrics import accuracy_score, classification_report, make_scorer, f1_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier
from scipy.stats import randint, uniform

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Warnings
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully!')


Libraries imported successfully!


## 2. Load Data

In [17]:
# Load configuration
with open('outputs/preprocessing_config.json', 'r') as f:
    config = json.load(f)

# Load features
with open('outputs/extracted_features.pkl', 'rb') as f:
    features = pickle.load(f)

X_train = features['X_train_pca']
X_val = features['X_val_pca']
y_train = features['y_train']
y_val = features['y_val']

CLASSES = config['classes']
NUM_CLASSES = config['num_classes']

print(f"📊 Data Loaded:")
print(f"   - Training samples: {X_train.shape[0]:,}")
print(f"   - Feature dimension: {X_train.shape[1]}")
print(f"   - Classes: {CLASSES}")

📊 Data Loaded:
   - Training samples: 4,524
   - Feature dimension: 3201
   - Classes: ['cbb', 'cbsd', 'cgm', 'cmd', 'healthy']


In [18]:
# Define cross-validation strategy
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Define scoring
scoring = {
    'accuracy': 'accuracy',
    'f1_macro': make_scorer(f1_score, average='macro')
}

print("📊 Cross-Validation Setup:")
print(f"   - Strategy: 5-Fold Stratified")
print(f"   - Scoring: Accuracy, F1-Macro")

📊 Cross-Validation Setup:
   - Strategy: 5-Fold Stratified
   - Scoring: Accuracy, F1-Macro


## 3. Hyperparameter Tuning: Logistic Regression

**Parameters to tune:**
- `C`: Regularization strength (inverse)
- `solver`: Optimization algorithm
- `max_iter`: Maximum iterations

In [19]:
print('='*60)
print('GPU Hyperparameter Search: XGBoost')
print('='*60)

# Device selection for Windows (no RAPIDS needed)
xgb_device = 'cuda'
try:
    import torch
    if not torch.cuda.is_available():
        xgb_device = 'cpu'
except Exception:
    xgb_device = 'cpu'

print(f'Using device: {xgb_device}')

param_dist_xgb = {
    'n_estimators': randint(150, 600),
    'max_depth': randint(4, 12),
    'learning_rate': uniform(0.01, 0.2),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'min_child_weight': randint(1, 10),
    'reg_alpha': uniform(0.0, 1.0),
    'reg_lambda': uniform(0.5, 2.0)
}

print('RandomizedSearchCV setup: n_iter=30, cv=3, scoring=f1_macro')


GPU Hyperparameter Search: XGBoost
Using device: cuda
RandomizedSearchCV setup: n_iter=30, cv=3, scoring=f1_macro


In [20]:
# Initialize XGBoost base model
xgb_base = XGBClassifier(
    objective='multi:softprob',
    num_class=NUM_CLASSES,
    tree_method='hist',
    device=xgb_device,
    eval_metric='mlogloss',
    random_state=42
)

start_time = time.time()

xgb_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist_xgb,
    n_iter=30,
    cv=3,
    scoring='f1_macro',
    n_jobs=1,
    verbose=2,
    random_state=42,
    return_train_score=True
)

xgb_search.fit(X_train, y_train)

xgb_search_time = time.time() - start_time
print(f'XGBoost search completed in {xgb_search_time:.2f}s')


Fitting 3 folds for each of 30 candidates, totalling 90 fits
[CV] END colsample_bytree=0.749816047538945, learning_rate=0.20014286128198325, max_depth=6, min_child_weight=8, n_estimators=338, reg_alpha=0.596850157946487, reg_lambda=1.3916655057071823, subsample=0.6399899663272012; total time=  26.5s
[CV] END colsample_bytree=0.749816047538945, learning_rate=0.20014286128198325, max_depth=6, min_child_weight=8, n_estimators=338, reg_alpha=0.596850157946487, reg_lambda=1.3916655057071823, subsample=0.6399899663272012; total time=  33.5s
[CV] END colsample_bytree=0.749816047538945, learning_rate=0.20014286128198325, max_depth=6, min_child_weight=8, n_estimators=338, reg_alpha=0.596850157946487, reg_lambda=1.3916655057071823, subsample=0.6399899663272012; total time=  47.7s
[CV] END colsample_bytree=0.7836995567863468, learning_rate=0.07674172222780437, max_depth=11, min_child_weight=8, n_estimators=280, reg_alpha=0.020584494295802447, reg_lambda=2.4398197043239884, subsample=0.93297705632

KeyboardInterrupt: 

In [ ]:
# Display XGBoost results
print('XGBoost - Best Parameters:')
print(xgb_search.best_params_)
print(f'Best CV F1-Score: {xgb_search.best_score_ * 100:.2f}%')

y_val_pred_xgb = xgb_search.predict(X_val)
xgb_val_acc = accuracy_score(y_val, y_val_pred_xgb)
xgb_val_f1 = f1_score(y_val, y_val_pred_xgb, average='macro')

print(f'Validation Accuracy: {xgb_val_acc * 100:.2f}%')
print(f'Validation F1-Score: {xgb_val_f1 * 100:.2f}%')


In [ ]:
# Plot top XGBoost configurations
results_xgb = pd.DataFrame(xgb_search.cv_results_).sort_values('mean_test_score', ascending=False).head(10)

plt.figure(figsize=(12, 6))
plt.barh(range(len(results_xgb)), results_xgb['mean_test_score'], color='#2E8B57')
plt.gca().invert_yaxis()
plt.xlabel('Mean CV F1-Score')
plt.ylabel('Top 10 Configurations')
plt.title('Top XGBoost Hyperparameter Configurations')
plt.tight_layout()
plt.savefig('outputs/xgb_hyperparameter_top10.png', dpi=300, bbox_inches='tight')
plt.show()

print('Saved: outputs/xgb_hyperparameter_top10.png')


## 4. Hyperparameter Tuning: SVM

**Parameters to tune:**
- `C`: Regularization parameter
- `gamma`: Kernel coefficient
- `kernel`: Kernel type

In [ ]:
print('SVM tuning skipped: replaced by GPU XGBoost tuning.')

In [ ]:
print('# Skipped')

In [ ]:
print('# Skipped')

In [ ]:
print('# Skipped')

In [ ]:
print('# Skipped')

## 5. Hyperparameter Tuning: Random Forest

**Parameters to tune (using RandomizedSearchCV for efficiency):**
- `n_estimators`: Number of trees
- `max_depth`: Maximum depth
- `min_samples_split`: Minimum samples to split
- `min_samples_leaf`: Minimum samples in leaf
- `max_features`: Features at each split

In [ ]:
print('Random Forest tuning skipped: replaced by GPU XGBoost tuning.')

In [ ]:
print('# Skipped')

In [ ]:
print('# Skipped')

In [ ]:
print('# Skipped')

## 6. Summary of Best Parameters

In [ ]:
# Save best parameters
best_params = {
    'xgboost': xgb_search.best_params_
}

with open('outputs/best_hyperparameters.json', 'w') as f:
    json.dump(best_params, f, indent=2)

print('Best hyperparameters saved to outputs/best_hyperparameters.json')


## 7. Retrain Models with Best Parameters

In [ ]:
print('='*60)
print('Retraining Optimized Model (XGBoost)')
print('='*60)

xgb_optimized = xgb_search.best_estimator_
xgb_optimized.fit(X_train, y_train)

y_val_pred_xgb_opt = xgb_optimized.predict(X_val)
xgb_opt_acc = accuracy_score(y_val, y_val_pred_xgb_opt)
xgb_opt_f1 = f1_score(y_val, y_val_pred_xgb_opt, average='macro')

print(f'Accuracy: {xgb_opt_acc * 100:.2f}%')
print(f'F1: {xgb_opt_f1 * 100:.2f}%')


In [ ]:
# Save optimized model
with open('outputs/xgboost_optimized.pkl', 'wb') as f:
    pickle.dump(xgb_optimized, f)

print('Saved: outputs/xgboost_optimized.pkl')


## 8. Performance Comparison: Before vs After Tuning

In [ ]:
# Load original training results
with open('outputs/training_results.json', 'r') as f:
    original_results = json.load(f)

best_baseline_acc = max(
    original_results['logistic_regression']['val_accuracy'],
    original_results['svm']['val_accuracy'],
    original_results['random_forest']['val_accuracy']
)

comparison_df = pd.DataFrame({
    'Model': ['Best Baseline (Notebook 04)', 'XGBoost (Notebook 05)'],
    'Validation Accuracy (%)': [best_baseline_acc * 100, xgb_opt_acc * 100],
    'Validation F1 (%)': [best_baseline_acc * 100, xgb_opt_f1 * 100]
})

print(comparison_df.round(2).to_string(index=False))


In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(10, 5))

models = comparison_df['Model']
acc_values = comparison_df['Validation Accuracy (%)']

bars = ax.bar(models, acc_values, color=['#999999', '#2E8B57'], edgecolor='black', alpha=0.8)
ax.set_ylabel('Validation Accuracy (%)')
ax.set_title('Baseline vs XGBoost Accuracy')
ax.set_ylim(0, 100)

for bar in bars:
    h = bar.get_height()
    ax.annotate(f'{h:.1f}%', xy=(bar.get_x() + bar.get_width()/2, h),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom')

plt.tight_layout()
plt.savefig('outputs/tuning_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print('Saved: outputs/tuning_comparison.png')


## 9. Save Tuning Results

In [ ]:
# Save tuning results
tuning_results = {
    'xgboost': {
        'best_params': xgb_search.best_params_,
        'best_cv_score': float(xgb_search.best_score_),
        'val_accuracy': float(xgb_opt_acc),
        'val_f1_score': float(xgb_opt_f1),
        'search_time': float(xgb_search_time),
        'search_type': 'RandomizedSearchCV',
        'device': xgb_device
    }
}

with open('outputs/tuning_results.json', 'w') as f:
    json.dump(tuning_results, f, indent=2)

print('Saved: outputs/tuning_results.json')


## 10. Summary

In [ ]:
print('='*60)
print('HYPERPARAMETER TUNING SUMMARY')
print('='*60)
print('Model tuned: XGBoost (GPU on Windows if CUDA available)')
print(f'Search time: {xgb_search_time:.2f}s')
print(f'Best CV F1: {xgb_search.best_score_ * 100:.2f}%')
print(f'Validation Accuracy: {xgb_opt_acc * 100:.2f}%')
print(f'Validation F1: {xgb_opt_f1 * 100:.2f}%')
print('Saved files:')
print(' - outputs/best_hyperparameters.json')
print(' - outputs/tuning_results.json')
print(' - outputs/xgboost_optimized.pkl')
print(' - outputs/tuning_comparison.png')
print('='*60)


---
## 📌 Next Steps

Proceed to **Notebook 06: Model Evaluation** to:
1. Generate detailed metrics for all models
2. Create confusion matrices
3. Plot ROC curves
4. Compare models comprehensively
5. Select the best model

---
*End of Notebook 05*